<a href="https://colab.research.google.com/github/FallingFrost/robotics-practice/blob/main/apple_counting_colab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 苹果计数 · Colab 训练

**这一步的目的**：先跑 10 个 epoch 量出 T4 的真实迭代速度，再决定要不要上付费平台。

**前置**：把 `apple_counting_code.zip` 放到 Google Drive 的 `MyDrive/apple_counting/` 下。

**数据**：不用手动上传。本 notebook 会先看 Drive 上有没有现成缓存，
没有就在 Colab 里下载原始数据重建（约 4 GB 下载 + 几分钟构建），建完存回 Drive。


In [1]:
#@title 1. 定位代码包 + 挂载 Drive（Drive 可选）
import pathlib

PROJ = pathlib.Path('/content/apple_counting')      # 代码解压到这儿

# 代码包在这几个位置里找；都不对就改这一行
ZIP_CANDIDATES = [
    PROJ / 'apple_counting_code.zip',               # 直接上传到 Colab 本机
    pathlib.Path('/content/apple_counting_code.zip'),
    pathlib.Path('/content/drive/MyDrive/apple_counting/apple_counting_code.zip'),
]

# Drive 只用来**持久化缓存**（2.4 GB）。挂不上也能训练，
# 只是每次新会话都得重新下载+构建，多花十几分钟。
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = pathlib.Path('/content/drive/MyDrive/apple_counting')
    DRIVE.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    print('⚠️ Drive 没挂上，缓存不会跨会话保留：', exc)

SRC_ZIP = next((p for p in ZIP_CANDIDATES if p.exists()), None)
print('代码包：', SRC_ZIP if SRC_ZIP else '未找到')
if DRIVE:
    print('Drive 目录：', DRIVE)
    print('当前内容：', sorted(p.name for p in DRIVE.iterdir()) or '（空）')


Mounted at /content/drive
代码包： /content/apple_counting/apple_counting_code.zip
Drive 目录： /content/drive/MyDrive/apple_counting
当前内容： ['cache']


In [2]:
#@title 2. 环境检查
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc
import sys, torch
print('python', sys.version.split()[0], '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import importlib.util, subprocess, sys
if importlib.util.find_spec('timm') is None:      # 用 subprocess 而不是 !pip，
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'],   # 这样缩进块里也是合法 Python
                   check=True)
import timm
print('timm', timm.__version__)


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
2
python 3.13.15 | torch 2.11.0+cu130 | cuda True
GPU: Tesla T4 15.6 GB
timm 1.0.30


In [3]:
#@title 3. 代码就位
import zipfile, shutil, pathlib

assert SRC_ZIP is not None, (
    '找不到代码包。把它放到下面任一路径，或改 cell 1 的 ZIP_CANDIDATES：\n  '
    + '\n  '.join(str(p) for p in ZIP_CANDIDATES))

# ⚠️ 这里**不能**整个 rmtree(PROJ)：代码包本身可能就放在 PROJ 里，
# 整删会把 zip 一起删掉，重跑时就没得解压了。
# 只清掉 zip 会覆盖的那两个目录。
for sub in ('common', 'Apple Counting'):
    p = PROJ / sub
    if p.exists():
        shutil.rmtree(p)
PROJ.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SRC_ZIP) as z:
    z.extractall(PROJ)
print(f'代码已从 {SRC_ZIP} 解压到 {PROJ}')
for p in sorted(PROJ.rglob('*.py')):
    print('  ', p.relative_to(PROJ))


代码已从 /content/apple_counting/apple_counting_code.zip 解压到 /content/apple_counting
   Apple Counting/src/build_cache.py
   Apple Counting/src/build_index.py
   Apple Counting/src/check_dataset.py
   Apple Counting/src/evaluate.py
   Apple Counting/src/gen_from_detection.py
   Apple Counting/src/infer.py
   Apple Counting/src/train.py
   common/__init__.py
   common/dataset.py
   common/engine.py
   common/image_utils.py
   common/index.py
   common/logging_utils.py
   common/metrics.py
   common/models.py
   common/paths.py


In [4]:
#@title 4. 数据缓存（没有就自动建）
import os, subprocess, sys, shutil, pathlib, time

APPLE = PROJ / 'Apple Counting'
DATA = APPLE / 'data'
CACHE = APPLE / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE = (DRIVE / 'cache') if DRIVE else None

# 训练真正需要的文件（原图不上传，只传这两个 .npy 大数组和索引）
NEED = [
    'counting_96_images.npy', 'counting_96_labels.npy',
    'counting_96_split.npy', 'counting_96_group.npy', 'counting_index.csv',
    'generated_96_images.npy', 'generated_96_labels.npy',
    'generated_96_split.npy', 'generated_96_group.npy', 'generated_index.csv',
]

def complete(d):
    return all((d / n).exists() for n in NEED)


def save_to_drive():
    """把缓存同步回 Drive。

    增量式：Drive 上已有同尺寸文件就跳过，所以可以反复调用、随时调用。
    **必须在每个构建步骤之后立刻调一次**——之前把它放在整段流程末尾，
    结果第二个构建崩了、cell 中断，前面辛苦建好的缓存一个字节都没存下来。
    """
    if not DRIVE_CACHE:
        print('    （没挂 Drive，跳过保存）')
        return
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
    copied = 0
    for p in sorted(CACHE.iterdir()):
        if not p.is_file() or p.suffix not in ('.npy', '.csv'):
            continue
        dst = DRIVE_CACHE / p.name
        if dst.exists() and dst.stat().st_size == p.stat().st_size:
            continue
        shutil.copy2(p, dst)
        copied += 1
        print(f'    → Drive  {p.name}  {p.stat().st_size/1e6:.0f} MB', flush=True)
    print(f'    {"已同步 " + str(copied) + " 个文件" if copied else "Drive 上已是最新"}')


if DRIVE_CACHE and complete(DRIVE_CACHE):
    print('Drive 上有完整缓存，复制到本地 SSD（约 2.4 GB，比直接读 Drive 快得多）…')
    t0 = time.time()
    shutil.copytree(DRIVE_CACHE, CACHE, dirs_exist_ok=True)
    print(f'  完成，用时 {time.time()-t0:.0f}s')
elif complete(CACHE):
    print('本地已有缓存，顺手同步一次到 Drive')
    save_to_drive()
else:
    print('缓存不存在，从头构建（一次性的，之后会存回 Drive）')
    BASE = 'https://conservancy.umn.edu/server/api/core/bitstreams'
    FILES = {
        'counting.tar.gz':  f'{BASE}/9a8d9477-4549-4896-974b-e824c5ab2d19/content',
        'detection.tar.gz': f'{BASE}/3ef26f04-6467-469b-9857-f443ffa1bb61/content',
        'test_data.zip':    f'{BASE}/7856047c-1f23-43ef-a50d-9327ec2ae0ff/content',
    }
    DL = pathlib.Path('/content/_dl'); DL.mkdir(exist_ok=True)
    for name, url in FILES.items():
        if (DL / name).exists() and (DL / name).stat().st_size > 1e8:
            print('  已有', name); continue
        print('  下载', name, flush=True)
        subprocess.run(['curl', '-L', '--retry', '5', '--retry-delay', '3', '-sS',
                        '-o', str(DL / name), url], check=True)
        print(f'    {name}  {(DL/name).stat().st_size/1e6:.0f} MB')

    DATA.mkdir(parents=True, exist_ok=True)
    print('  解压…', flush=True)
    subprocess.run(['tar', '-xzf', str(DL/'counting.tar.gz'),  '-C', str(DATA)], check=True)
    subprocess.run(['tar', '-xzf', str(DL/'detection.tar.gz'), '-C', str(DATA)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(DL/'test_data.zip'), '-d', str(DATA)], check=True)

    def run_step(title, script, extra=()):
        """跑一个构建脚本，**失败时把子进程输出整段打出来**。

        不能直接用 check=True：那样只会抛一个 CalledProcessError，
        真正的 traceback 在子进程里，看不到就没法排查。
        """
        cmd = [sys.executable, str(APPLE / 'src' / script), *extra]
        print(f'  {title}…', flush=True)
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJ))
        if r.returncode != 0:
            print(f'\n!!! {title} 失败（exit {r.returncode}）')
            print('---- stdout ----')
            print(r.stdout[-8000:] if r.stdout else '(空)')
            print('---- stderr ----')
            print(r.stderr[-8000:] if r.stderr else '(空)')
            raise SystemExit(
                f'{title} 失败。完整输出见上；也可用 colab/diagnose_gen.py 单独诊断。')
        tail = [ln for ln in (r.stdout or '').splitlines() if ln.strip()][-3:]
        for ln in tail:
            print('    ', ln)
        print(f'    {title} 完成')

    # 索引表已随代码包带上（counting_index.csv / detection_index.csv），
    # 省掉在 Colab 上重算 7 万个 MD5 的几分钟。
    #
    # 每建完一步就立刻存 Drive —— 不要等全部做完再存。
    # 之前就是攒到最后一起存，第二个构建崩了导致全部白干。
    try:
        run_step('构建图块缓存', 'build_cache.py', ('--workers', '2'))
        save_to_drive()
        run_step('从掩膜生成计数图块', 'gen_from_detection.py', ('--workers', '2'))
    finally:
        save_to_drive()          # 即使中途失败，已建好的部分也保住
        shutil.rmtree(DL, ignore_errors=True)

print('\n缓存文件：')
for n in NEED:
    p = CACHE / n
    print(f'  {n:<32} {p.stat().st_size/1e6:9.1f} MB' if p.exists() else f'  {n}  缺失!')


Drive 上有完整缓存，复制到本地 SSD（约 2.4 GB，比直接读 Drive 快得多）…
  完成，用时 47s

缓存文件：
  counting_96_images.npy              1959.3 MB
  counting_96_labels.npy                 0.1 MB
  counting_96_split.npy                  0.1 MB
  counting_96_group.npy                  0.1 MB
  counting_index.csv                    12.3 MB
  generated_96_images.npy              580.6 MB
  generated_96_labels.npy                0.0 MB
  generated_96_split.npy                 0.0 MB
  generated_96_group.npy                 0.0 MB
  generated_index.csv                    1.5 MB


In [7]:
! nvidia-smi

Tue Oct 13 01:55:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             32W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
#@title 5. 训练（计时）
%cd /content/apple_counting

MODEL = 'convnextv2_nano'   #@param ['convnextv2_nano','convnextv2_tiny','convnextv2_atto','resnet18','convnext_tiny_dinov3']
EPOCHS = 10                 #@param {type:'integer'}
BATCH = 128                 #@param {type:'integer'}
RUN = f'colab_t4_{MODEL}_{EPOCHS}ep'

# 写成一行：Colab 的 `!` 命令里用反斜杠续行有时会出问题
!python "Apple Counting/src/train.py" --model {MODEL} --epochs {EPOCHS} --batch-size {BATCH} --num-workers 2 --run-name {RUN}


/content/apple_counting
2026-10-13 00:17:33 - root - INFO : ==============训练 colab_t4_convnextv2_nano_10ep==============
2026-10-13 00:17:33 - root - INFO :   设备 cuda   AMP=True
2026-10-13 00:17:33 - root - INFO :   GPU Tesla T4
2026-10-13 00:17:33 - root - INFO :   convnextv2_nano  输入 224  来源 convnextv2_nano.fcmae_ft_in1k  
2026-10-13 00:17:37 - timm.models._builder - INFO : Loading pretrained weights from Hugging Face hub (timm/convnextv2_nano.fcmae_ft_in1k)
2026-10-13 00:17:38 - httpx - INFO : HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-13 00:17:38 - httpx - INFO : HTTP Request: HEAD https://huggingface.co/timm/convnextv2_nano.fcmae_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-10-13 00:17:38 - httpx - INFO : HTTP Request: GET https://huggingface.co/api/models/timm/convnextv2_nano.fcmae_ft_in1k/xet-read-token/fd83d9c986153f1bac0fc78368a17a66eb9c584e "HTTP/1.1 200 OK"

model.safetensors: downloading bytes:  23% 14.6M/62.5M

In [6]:
#@title 6. 速度汇总与决策依据
import pandas as pd, pathlib, json

run_dir = pathlib.Path(f'/content/apple_counting/Apple Counting/runs/{RUN}')
hist = pd.read_csv(run_dir / 'history.csv')
sec = hist['seconds'].astype(float)

print('=== 每轮耗时（秒）===')
print(sec.describe()[['mean', 'min', 'max']].to_string())
print(f'\n前 3 轮均值 {sec.head(3).mean():.1f}s  后 3 轮均值 {sec.tail(3).mean():.1f}s')
print('（后段比前段慢说明有热降频或数据加载瓶颈）')

per = sec.mean()
print(f'\n估算：60 轮 ≈ {per*60/60:.1f} 分钟')
print('换平台的话，按纸面峰值算的下限参考（实际提速通常低于这个数，')
print('因为小模型吃不满峰值算力，瓶颈常在显存带宽和 kernel 启动开销）：')
for name, mult in [('L4', 1.6), ('V100', 1.8), ('A100 40G', 3.5), ('RTX 4090', 4.0)]:
    print(f'  {name:<10} 最快约 {per*60/mult/60:5.1f} 分钟（按 {mult}× T4 峰值估）')
print('拿这个数去对比各平台的时租，再看哪个划算。')

print('\n=== 最后一轮的指标 ===')
row = hist.iloc[-1]
print(f"  train_loss {row['train_loss']:.4f}  train_acc {row['train_acc']:.4f}")
print(f"  val 精确匹配 {row['val_acc']:.4f} / EMA {row['val_acc_ema']:.4f}")
print(f"  val 先验加权 {row['val_pw_acc']:.4f} / EMA {row['val_pw_acc_ema']:.4f}")

# 训练曲线和权重落盘，方便回本地看
import shutil
if DRIVE:
    shutil.copy(run_dir / 'history.csv', DRIVE / f'{RUN}_history.csv')
    shutil.copy(run_dir / 'best.pt', DRIVE / f'{RUN}_best.pt')
    print(f'\n曲线和权重已存回 Drive：{RUN}_history.csv / {RUN}_best.pt')
else:
    print(f'\n没挂 Drive。产物在 {run_dir}')
    print('  要带走的话：从左侧文件面板右键下载 history.csv')
    print('  （best.pt 有 27 MB，测速用不上，可以不下载）')
print('\n10 轮只是测速，成绩不完整。要拿最终成绩请跑满 60 轮并加 --eval-test。')


=== 每轮耗时（秒）===
mean    560.92
min     558.10
max     566.90

前 3 轮均值 562.4s  后 3 轮均值 559.3s
（后段比前段慢说明有热降频或数据加载瓶颈）

估算：60 轮 ≈ 560.9 分钟
换平台的话，按纸面峰值算的下限参考（实际提速通常低于这个数，
因为小模型吃不满峰值算力，瓶颈常在显存带宽和 kernel 启动开销）：
  L4         最快约 350.6 分钟（按 1.6× T4 峰值估）
  V100       最快约 311.6 分钟（按 1.8× T4 峰值估）
  A100 40G   最快约 160.3 分钟（按 3.5× T4 峰值估）
  RTX 4090   最快约 140.2 分钟（按 4.0× T4 峰值估）
拿这个数去对比各平台的时租，再看哪个划算。

=== 最后一轮的指标 ===
  train_loss 0.4077  train_acc 0.9350
  val 精确匹配 0.9299 / EMA 0.8919
  val 先验加权 0.9029 / EMA 0.9129

曲线和权重已存回 Drive：colab_t4_convnextv2_nano_10ep_history.csv / colab_t4_convnextv2_nano_10ep_best.pt

10 轮只是测速，成绩不完整。要拿最终成绩请跑满 60 轮并加 --eval-test。


## 决策参考

拿到上面的「每轮耗时」后，按这个看：

| 现象 | 含义 |
|---|---|
| 后 3 轮比前 3 轮明显慢 | Colab 热降频，付费平台的稳定 GPU 优势更大 |
| 每轮时间波动大 | 数据加载跟不上（Colab 只有 2 核），加钱买核心数比买显卡更值 |
| 显存不够（OOM） | 降 `BATCH` 到 64，或换更小的模型 |
| 10 轮后 val 先验加权还在涨 | 正常，60 轮会更好，值得上付费平台 |
| 10 轮后已经平了 | 先别急着加钱，先查是不是学习率或增强的问题 |
